In [6]:
# Install required packages (Run only once)

import torch
import transformers
import datasets

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

Torch: 2.13.0+cu130
Transformers: 5.14.1
Datasets: 5.0.1


In [7]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score
import torch

Create Custom Dataset

In [8]:
data = [
    {"text": "I love this product", "label": 1},
    {"text": "This is amazing", "label": 1},
    {"text": "Very bad experience", "label": 0},
    {"text": "I hate this", "label": 0},
    {"text": "Absolutely fantastic", "label": 1},
    {"text": "Worst service ever", "label": 0},
]

dataset = Dataset.from_list(data)

dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 6
})

Split Dataset

In [9]:
dataset = dataset.train_test_split(test_size=0.3)

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2
    })
})

Load Tokenizer

In [10]:

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

Tokenize Dataset

In [11]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length"
    )

dataset = dataset.map(tokenize)

dataset = dataset.rename_column("label", "labels")

dataset.set_format("torch")

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Load Pre-trained Model

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Define Training Arguments

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="no"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Define Evaluation Metrics

In [14]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = logits.argmax(axis=1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

Train Model

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.686519,0.500000,0.000000
2,No log,0.681946,0.500000,0.000000
3,No log,0.679887,0.500000,0.000000


TrainOutput(global_step=6, training_loss=0.6810526847839355, metrics={'train_runtime': 1.2319, 'train_samples_per_second': 9.741, 'train_steps_per_second': 4.871, 'total_flos': 1589608783872.0, 'train_loss': 0.6810526847839355, 'epoch': 3.0})

Evaluate Model

In [17]:
results = trainer.evaluate()

print("Evaluation Results")

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.679887,3,0.500000,0.000000


Evaluation Results
{'eval_loss': 0.6798874735832214, 'eval_accuracy': 0.5, 'eval_f1': 0.0}


Test with New Sentence

In [19]:
text = "This product is really good"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move inputs to the same device as the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(device) for k, v in inputs.items()}
model.to(device)

outputs = model(**inputs)

prediction = outputs.logits.argmax().item()

print("Sentence:", text)

print("Prediction:",
      "Positive" if prediction == 1 else "Negative")

Sentence: This product is really good
Prediction: Negative


Save Fine-Tuned Model

In [20]:
model.save_pretrained("fine_tuned_model")

tokenizer.save_pretrained("fine_tuned_model")

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!
